# Music Annotation Project V9.0B
## Notebook 60R2: Fresh SAFE_REMOVE inference on reviewed HITL masks

**Runtime:** CPU. In Google Colab choose **Runtime -> Run all**.

60R2 revises 60R1 to consume the Notebook 59R1 deployment bundle. The `.joblib` files are loaded from the frozen Drive bundle and are not embedded in this notebook.

### Fixed rules
- Notebook 07 aligned observed page and reviewed/corrected mask are the page authority.
- No clean page, training, fitting, recalibration, threshold movement or SAFE_PROTECT.
- Construct the exact 17 predictors in `predictor_schema.json` order.
- Frozen REMOVE threshold 0.995, page-global 8-connectivity, minimum run 2, region mean 0.995, every region pixel 0.995.
- `M5_REMOVE = M3 & ~SAFE_REMOVE`; all unsupported/invalid/failed coordinates retain exact M3.
- No pixel outside the prepared authoritative mask may change.
- Research/HITL evidence only. Production authority remains false.

**Important:** Notebook06 YOLO confidence is audit-only and is never used as `p_print`. `p_print` must come from a separately frozen all-development observed-print deployment bundle. The historical V8.2 OOF folds are not silently ensembled.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow scikit-learn joblib matplotlib


## 1. Configuration and helpers


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, sys, platform, time, shutil, traceback, zipfile
import cv2, joblib, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
DRAFT_AUDIT_DIR=ROOT/'human_in_the_middle'/'draft_audit'
REVIEW_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'
N59_ROOT=ROOT/'observable_residual_mask_m5_v8_8_deployment_freeze'
NOTEBOOK59_RUN_OVERRIDE=None
OUTPUT_ROOT=ROOT/'v9_0b_notebook60r2_fresh_safe_remove'
PAGE_ALLOWLIST=None; EXPECTED_REVIEWED_PAGES=151; REQUIRE_EXACT_151=False; STOP_AT_FIRST_PAGE_FAILURE=False; MAX_VISUAL_PAGES=30
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook60r2_v9_0b'; OUT=OUTPUT_ROOT/RUN
INPUTS=OUT/'inputs'; PAGES=OUT/'pages'; TABLES=OUT/'tables'; AUDIT=OUT/'audit'; FAILURES=OUT/'failures'; VISUAL=OUT/'visual'; UPLOAD=OUT/'upload'
for d in [OUT,INPUTS,PAGES,TABLES,AUDIT,FAILURES,VISUAL,UPLOAD]: d.mkdir(parents=True,exist_ok=False)
def require(x,msg):
    if not x: raise RuntimeError(msg)
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def atomic_json(p,v):
    t=Path(str(p)+'.tmp'); t.write_text(json.dumps(v,indent=2,sort_keys=True,default=str)); t.replace(p)
def read_json(p): return json.loads(Path(p).read_text())
def read_gray(p):
    a=cv2.imread(str(p),cv2.IMREAD_GRAYSCALE); require(a is not None,f'Unreadable image: {p}'); return a


## 2. Resolve and verify the Notebook 59R1 deployment bundle


In [ ]:
REQUIRED=['remove_model.joblib','remove_calibrator.joblib','predictor_schema.json','policy.json','reason_code_schema.json','freeze_manifest.json','runtime_contract.json','replay_test.json']
def bundles(root): return sorted([p for p in root.rglob('bundle') if p.is_dir() and all((p/x).exists() for x in REQUIRED)],key=lambda p:p.stat().st_mtime)
if NOTEBOOK59_RUN_OVERRIDE:
    q=Path(NOTEBOOK59_RUN_OVERRIDE); BUNDLE=q/'bundle' if (q/'bundle').exists() else q
else:
    hit=bundles(N59_ROOT); require(hit,'No complete Notebook 59R1 bundle found.'); BUNDLE=hit[-1]
policy=read_json(BUNDLE/'policy.json'); schema=read_json(BUNDLE/'predictor_schema.json'); reasons=read_json(BUNDLE/'reason_code_schema.json'); freeze=read_json(BUNDLE/'freeze_manifest.json'); contract=read_json(BUNDLE/'runtime_contract.json'); replay=read_json(BUNDLE/'replay_test.json')
checks={'deployment_bundle_complete':freeze.get('deployment_bundle_complete') is True,'replay_passed':replay.get('passed') is True,'policy_id_match':policy.get('policy_id')==freeze.get('policy_id')==contract.get('policy_id')==replay.get('policy_id'),'compatibility_match':policy.get('compatibility_id')==freeze.get('compatibility_id')==contract.get('compatibility_id')==replay.get('compatibility_id'),'safe_protect_disabled':policy.get('safe_protect_enabled') is False and freeze.get('protect_included') is False,'promotion_false':policy.get('promotion_authority') is False and freeze.get('independent_promotion_authority') is False,'schema_17':schema.get('predictor_count')==17 and len(schema.get('ordered_predictors',[]))==17}
for o in freeze.get('objects',[]): checks['hash:'+o['relative_path']]=(BUNDLE/o['relative_path']).exists() and sha256_file(BUNDLE/o['relative_path'])==o['sha256']
require(all(checks.values()),f'59R1 bundle integrity failed: {checks}')
model=joblib.load(BUNDLE/policy['model_file']); calibrator=joblib.load(BUNDLE/policy['calibrator_file']); predictors=schema['ordered_predictors']
atomic_json(AUDIT/'notebook59r1_source.json',{'bundle':str(BUNDLE),'checks':checks,'policy_id':policy['policy_id'],'compatibility_id':policy['compatibility_id'],'model_sha256':sha256_file(BUNDLE/policy['model_file']),'calibrator_sha256':sha256_file(BUNDLE/policy['calibrator_file']),'schema_sha256':sha256_file(BUNDLE/'predictor_schema.json')})
print('59R1 bundle:',BUNDLE); print('Policy:',policy['policy_id']); print('Integrity: PASS')


## Observed-print deployment bundle and fresh `p_print` inference


In [ ]:

# ============================================================
# OBSERVED-PRINT p_print DEPLOYMENT BUNDLE
# ============================================================
#
# PROVENANCE:
# V8.2 development trained source-group OOF 33x33 CNN fold models and
# persisted fold-specific isotonic calibrators. Those are development OOF
# objects, not a registered new-page deployment ensemble.
#
# This cell therefore requires one separately frozen all-development
# observed-print bundle. It does NOT average the historical OOF folds and
# does NOT substitute Notebook06 YOLO confidence for p_print.
#
# Expected bundle contents:
#   model.pt
#   calibrator.joblib
#   channel_schema.json
#   policy.json
#   freeze_manifest.json
#   replay/replay_test.json   OR replay_test.json
#
# The bundle must describe the unchanged V8.2 compact TopologyCNN,
# 33x33 patch, 11 ordered observable channels, and the registered
# PRINT/ANNOTATION/UNCERTAIN policy.
# ============================================================

import torch
from torch import nn

# Point this to the ACTUAL frozen all-development observed-print bundle.
# Leave None to fail closed. Do not point this to the V8.2 OOF run itself.
OBSERVED_PRINT_BUNDLE_OVERRIDE = None

OBSERVED_PRINT_REQUIRED = [
    'model.pt',
    'calibrator.joblib',
    'channel_schema.json',
    'policy.json',
    'freeze_manifest.json',
]


def resolve_observed_print_bundle(override):
    require(
        override is not None,
        (
            'No frozen all-development observed-print deployment bundle is configured. '
            'The historical V8.2 run contains source-group OOF fold models/calibrators, '
            'not a registered fresh-page deployment model. Create/freeze that bundle '
            'before running fresh 60R2 SAFE_REMOVE inference. '
            'Do not substitute Notebook06 YOLO confidence or average the OOF folds.'
        ),
    )

    bundle = Path(override)
    require(
        bundle.exists() and bundle.is_dir(),
        f'Observed-print bundle does not exist: {bundle}',
    )

    missing = [name for name in OBSERVED_PRINT_REQUIRED if not (bundle / name).exists()]
    require(
        not missing,
        f'Observed-print deployment bundle is incomplete: {missing}',
    )

    replay_candidates = [
        bundle / 'replay_test.json',
        bundle / 'replay' / 'replay_test.json',
        bundle.parent / 'replay' / 'replay_test.json',
    ]
    replay_hits = [p for p in replay_candidates if p.exists() and p.is_file()]
    require(
        len({str(p.resolve()) for p in replay_hits}) == 1,
        f'Expected exactly one observed-print replay_test.json; found {replay_hits}',
    )
    replay_path = replay_hits[0]

    return bundle, replay_path


PPRINT_BUNDLE, PPRINT_REPLAY_PATH = resolve_observed_print_bundle(
    OBSERVED_PRINT_BUNDLE_OVERRIDE
)

pprint_schema = read_json(PPRINT_BUNDLE / 'channel_schema.json')
pprint_policy = read_json(PPRINT_BUNDLE / 'policy.json')
pprint_freeze = read_json(PPRINT_BUNDLE / 'freeze_manifest.json')
pprint_replay = read_json(PPRINT_REPLAY_PATH)

PPRINT_CHANNELS = (
    pprint_schema.get('channels')
    or pprint_schema.get('ordered_channels')
)
PPRINT_PATCH = int(pprint_schema.get('patch', pprint_schema.get('patch_size', 33)))

EXPECTED_PPRINT_CHANNELS = [
    'obs', 'norm', 'mask', 'depth',
    'horizontal', 'vertical', 'compact',
    'preserve', 'remove', 'ambiguous', 'edge',
]

require(
    PPRINT_CHANNELS == EXPECTED_PPRINT_CHANNELS,
    f'Observed-print channel schema mismatch: {PPRINT_CHANNELS}',
)
require(PPRINT_PATCH == 33, f'Observed-print patch size must be 33; found {PPRINT_PATCH}')
require(pprint_replay.get('passed') is True, 'Observed-print deployment replay did not pass.')
require(pprint_policy.get('promotion_authority') is not True,
        'Observed-print bundle unexpectedly claims automatic promotion authority.')

# Verify freeze-manifest hashes for recorded objects.
for item in pprint_freeze.get('objects', []):
    rel = item.get('relative_path')
    digest = item.get('sha256')
    require(rel and digest, f'Invalid observed-print freeze object: {item}')
    path = PPRINT_BUNDLE / rel
    require(path.exists(), f'Observed-print frozen object missing: {path}')
    require(sha256_file(path) == digest, f'Observed-print frozen-object hash mismatch: {path}')


class TopologyCNN(nn.Module):
    def __init__(self, nc=11):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(nc, 24, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(24),
            nn.MaxPool2d(2),
            nn.Conv2d(24, 48, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(48),
            nn.MaxPool2d(2),
            nn.Conv2d(48, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(64, 2)

    def forward(self, x):
        return self.head(self.net(x).flatten(1))


PPRINT_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
pprint_model = TopologyCNN(len(PPRINT_CHANNELS)).to(PPRINT_DEVICE)
loaded = torch.load(PPRINT_BUNDLE / 'model.pt', map_location=PPRINT_DEVICE)
if isinstance(loaded, dict) and 'state_dict' in loaded:
    loaded = loaded['state_dict']
if isinstance(loaded, dict) and 'model_state_dict' in loaded:
    loaded = loaded['model_state_dict']
require(isinstance(loaded, dict), 'Observed-print checkpoint does not contain a state_dict.')
loaded = {str(k).removeprefix('module.'): v for k, v in loaded.items()}
pprint_model.load_state_dict(loaded, strict=True)
pprint_model.eval()
pprint_calibrator = joblib.load(PPRINT_BUNDLE / 'calibrator.joblib')
require(hasattr(pprint_calibrator, 'predict'), 'Observed-print calibrator lacks predict().')

PPRINT_HALF = PPRINT_PATCH // 2
PPRINT_BATCH = int(pprint_policy.get('batch_size', 512))


def observed_print_channel_arrays(observed, prepared_mask, routes):
    mask = prepared_mask > 0
    paper = routes['paper']
    norm = routes.get('normalised', routes.get('norm'))
    require(norm is not None, 'Frozen V7.8 normalised channel is missing.')
    arrays = {
        'obs': observed.astype(np.float32) / 255.0,
        'norm': norm.astype(np.float32) / 255.0,
        'mask': mask.astype(np.float32),
        'depth': cv2.distanceTransform(mask.astype(np.uint8), cv2.DIST_L2, 5).astype(np.float32),
        'horizontal': routes['horizontal'].astype(np.float32),
        'vertical': routes['vertical'].astype(np.float32),
        'compact': routes['compact'].astype(np.float32),
        'preserve': routes['preserve'].astype(np.float32),
        'remove': routes['remove'].astype(np.float32),
        'ambiguous': routes['ambiguous'].astype(np.float32),
        'edge': (cv2.Canny(observed, 70, 170) > 0).astype(np.float32),
    }
    require(all(a.shape == observed.shape for a in arrays.values()),
            'Observed-print channel shape mismatch.')
    return arrays


def score_fresh_p_print(observed, prepared_mask, routes):
    """Return a full-page calibrated p_print array.

    The registered eligible population is dark pixels inside the prepared
    authoritative mask that V7.8 does not already preserve. Other coordinates
    are left at zero because Notebook59R1 only consumes p_print on its M3 rows.
    """
    arrays = observed_print_channel_arrays(observed, prepared_mask, routes)
    eligible = (prepared_mask > 0) & (observed < 200) & ~routes['preserve']
    ys, xs = np.where(eligible)
    pmap = np.zeros(observed.shape, dtype=np.float32)

    if len(ys) == 0:
        return pmap, eligible

    padded = {
        key: np.pad(arrays[key], PPRINT_HALF, mode='reflect')
        for key in PPRINT_CHANNELS
    }

    values = []
    with torch.no_grad():
        for start in range(0, len(ys), PPRINT_BATCH):
            by = ys[start:start + PPRINT_BATCH]
            bx = xs[start:start + PPRINT_BATCH]
            batch = np.stack([
                np.stack([
                    padded[key][y:y + PPRINT_PATCH, x:x + PPRINT_PATCH]
                    for key in PPRINT_CHANNELS
                ])
                for y, x in zip(by, bx)
            ]).astype(np.float32)
            logits = pprint_model(torch.from_numpy(batch).to(PPRINT_DEVICE))
            raw = torch.softmax(logits, dim=1)[:, 1].cpu().numpy().astype(np.float64)
            calibrated = np.asarray(pprint_calibrator.predict(raw), dtype=np.float64)
            require(np.isfinite(calibrated).all(), 'Observed-print calibrator emitted non-finite values.')
            require(((calibrated >= 0.0) & (calibrated <= 1.0)).all(),
                    'Observed-print calibrated probability outside [0,1].')
            values.append(calibrated)

    values = np.concatenate(values).astype(np.float32)
    require(len(values) == len(ys), 'Observed-print scorer row-count mismatch.')
    pmap[ys, xs] = values
    return pmap, eligible

atomic_json(
    AUDIT / 'observed_print_bundle_source.json',
    {
        'bundle': str(PPRINT_BUNDLE),
        'replay_test': str(PPRINT_REPLAY_PATH),
        'model_sha256': sha256_file(PPRINT_BUNDLE / 'model.pt'),
        'calibrator_sha256': sha256_file(PPRINT_BUNDLE / 'calibrator.joblib'),
        'schema_sha256': sha256_file(PPRINT_BUNDLE / 'channel_schema.json'),
        'policy_sha256': sha256_file(PPRINT_BUNDLE / 'policy.json'),
        'channels': PPRINT_CHANNELS,
        'patch': PPRINT_PATCH,
        'device': PPRINT_DEVICE,
    },
)

print('Observed-print deployment bundle:', PPRINT_BUNDLE)
print('Observed-print replay:', PPRINT_REPLAY_PATH)
print('Observed-print device:', PPRINT_DEVICE)


## 3. Canonical V7.8/M3 preparation and 17 frozen predictors


In [ ]:
MASK_DILATION=2; CROP_MARGIN=48; DARK_THRESHOLD=200
def immutable(original,candidate,mask):
    out=original.copy(); out[mask>0]=candidate[mask>0]; require(np.array_equal(out[mask==0],original[mask==0]),'Outside-mask drift'); return out
def prepare_mask(seed): return cv2.dilate((seed>0).astype(np.uint8)*255,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5)))
def telea(observed,mask): return immutable(observed,cv2.inpaint(observed,mask,3,cv2.INPAINT_TELEA),mask)
def background(observed,mask,sigma=45.0):
    visible=(mask==0)&(observed>=200); num=cv2.GaussianBlur(observed.astype(np.float32)*visible,(0,0),sigma); den=cv2.GaussianBlur(visible.astype(np.float32),(0,0),sigma); fallback=cv2.GaussianBlur(observed,(0,0),sigma); return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)
def support_maps(observed,mask,band=4):
    dark=(observed<190)&(mask==0); h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0; v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0; n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8); c=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,hg,a=stats[i]
        if 3<=a<=650 and w<=40 and hg<=40:c[lab==i]=True
    k=np.ones((2*band+1,2*band+1),np.uint8); return *(cv2.dilate(z.astype(np.uint8),k)>0 for z in [h,v,c]),
def geometry(observed,mask,t):
    paper=background(observed,mask); norm=cv2.divide(observed,np.maximum(paper,1),scale=255); horizontal,vertical,compact=support_maps(observed,mask); m=mask>0; preserve=m&(horizontal|vertical|compact)&(observed<200); remove=m&~preserve&(observed>=80)&(norm>=80); ambiguous=m&~preserve&~remove; result=observed.copy(); result[remove]=paper[remove]; result[ambiguous]=t[ambiguous]; result[preserve]=observed[preserve]; return immutable(observed,result,mask),{'paper':paper,'norm':norm,'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':horizontal,'vertical':vertical,'compact':compact}
def frac(a,r=3): return cv2.boxFilter(a.astype(np.float32),-1,(2*r+1,2*r+1),normalize=True,borderType=cv2.BORDER_REFLECT)
def distance_to(a): return cv2.distanceTransform((~a).astype(np.uint8),cv2.DIST_L2,5).astype(np.float32) if a.any() else np.full(a.shape,float(max(a.shape)),np.float32)
def predictor_frame(observed,m3,routes,p_print):
    edge=cv2.Canny(observed,70,170)>0; boundary=m3^(cv2.erode(m3.astype(np.uint8),np.ones((3,3),np.uint8))>0); protected=routes['preserve']; _,components=cv2.connectedComponents(m3.astype(np.uint8),8); ys,xs=np.where(m3)
    d={'observed_value':observed,'normalised_value':routes['norm'],'paper_value':routes['paper'],'mask_depth':cv2.distanceTransform(m3.astype(np.uint8),cv2.DIST_L2,5),'boundary_distance':distance_to(boundary),'distance_to_protected':distance_to(protected),'edge':edge,'local_edge_fraction':frac(edge),'local_dark_fraction':frac(observed<190),'local_preserve_fraction':frac(routes['preserve']),'local_remove_fraction':frac(routes['remove']),'local_ambiguous_fraction':frac(routes['ambiguous']),'local_horizontal_fraction':frac(routes['horizontal']),'local_vertical_fraction':frac(routes['vertical']),'local_compact_fraction':frac(routes['compact']),'p_print':p_print,'component_id':components}
    frame=pd.DataFrame({'y':ys,'x':xs,**{k:np.asarray(v)[ys,xs] for k,v in d.items()}}); require(list(frame[predictors].columns)==predictors,'Predictor order mismatch'); X=frame[predictors].apply(pd.to_numeric,errors='coerce'); require(not np.isinf(X.to_numpy(float)).any(),'Infinite predictor'); return frame,X


## 4. Resolve Notebook 06 -> 07 reviewed HITL cohort


In [ ]:
def read_audits(folder,label):
    good=[]; bad=[]
    for p in sorted(Path(folder).glob('*.json')):
        try:d=read_json(p); require(isinstance(d,dict),'root not object'); d['_audit_path']=str(p); d['_type']=label; good.append(d)
        except Exception as e:bad.append({'type':label,'path':str(p),'error':str(e)})
    return good,bad
def path_from(r,keys,required=True):
    for k in keys:
        if r.get(k) and Path(r[k]).exists():return Path(r[k]),k
    if required:raise RuntimeError(f"{r.get('_audit_path')}: no path for {keys}")
    return None,None
reviews,rf=read_audits(REVIEW_AUDIT_DIR,'review'); drafts,df=read_audits(DRAFT_AUDIT_DIR,'draft'); pd.DataFrame(rf+df).to_csv(AUDIT/'audit_parse_failures.csv',index=False); require(not(rf+df),'Unreadable HITL audits')
drafts_by_md5={}
for d in drafts:
    if d.get('source_md5'):drafts_by_md5.setdefault(str(d['source_md5']).lower(),[]).append(d)
rows=[]; unresolved=[]
for r in reviews:
    try:
        observed,_=path_from(r,['aligned_output','aligned_page','observed_output']); mask,_=path_from(r,['corrected_mask_output','corrected_mask','mask_output']); h=str(r.get('source_md5','')).lower(); ds=drafts_by_md5.get(h,[]); draft=ds[0] if len(ds)==1 else None; confidence=None
        if draft: confidence,_=path_from(draft,['confidence_map','confidence_map_output'],required=False)
        o=read_gray(observed); m=read_gray(mask); require(o.shape==m.shape,'shape mismatch'); c=read_gray(confidence); require(c.shape==o.shape,'confidence shape mismatch')
        rows.append({'page':Path(observed).stem,'observed_path':str(observed),'reviewed_mask_path':str(mask),'confidence_map_path':str(confidence),'review_audit_path':r['_audit_path'],'draft_audit_path':draft['_audit_path'] if draft else None})
    except Exception as e:unresolved.append({'review_audit_path':r.get('_audit_path'),'error':str(e)})
roster=pd.DataFrame(rows)
if PAGE_ALLOWLIST is not None:roster=roster[roster.page.isin(set(PAGE_ALLOWLIST))]
require(not roster.empty,'No reviewed pages resolved'); require(not roster.page.duplicated().any(),'Duplicate pages'); roster=roster.sort_values('page').reset_index(drop=True); roster.to_csv(TABLES/'hitl_reviewed_roster.csv',index=False); pd.DataFrame(unresolved).to_csv(TABLES/'unresolved_review_lineages.csv',index=False)
if REQUIRE_EXACT_151:require(len(roster)==EXPECTED_REVIEWED_PAGES,f'Expected 151, resolved {len(roster)}')
print('Resolved:',len(roster),'Unresolved:',len(unresolved))


## 5. Frozen REMOVE scoring, M5 reconstruction and fail-closed pages


In [ ]:
THRESH=float(policy['remove_probability_threshold']); MIN_RUN=int(policy['minimum_remove_run']); MIN_MEAN=float(policy['remove_region_minimum_mean_probability']); MIN_FRAC=float(policy['remove_region_minimum_pixel_fraction_above_threshold']); CONN=int(policy['connectivity'])
page_rows=[]; failures=[]; parts=[]
for pos,r in roster.iterrows():
    page=str(r.page); d=PAGES/page; t0=time.time()
    try:
        observed=read_gray(r.observed_path); seed=read_gray(r.reviewed_mask_path); p_print, p_print_eligible = score_fresh_p_print(observed, authority, routes); authority=prepare_mask(seed); t=telea(observed,authority); baseline,routes=geometry(observed,authority,t); m3=authority>0
        frame,X=predictor_frame(observed,m3,routes,p_print); raw=model.predict_proba(X)[:,1].astype(float); p=np.asarray(calibrator.predict(raw),float); require(np.isfinite(p).all(),'non-finite probabilities'); provisional=p>=THRESH
        pm=np.zeros(observed.shape,np.uint8); pm[frame.y,frame.x]=provisional.astype(np.uint8); n,lab,stats,_=cv2.connectedComponentsWithStats(pm,CONN); safe=np.zeros(observed.shape,bool); accepted=[]
        pmap=np.full(observed.shape,np.nan,np.float32); pmap[frame.y,frame.x]=p
        for cid in range(1,n):
            region=lab==cid; vals=pmap[region]; ok=int(region.sum())>=MIN_RUN and float(vals.mean())>=MIN_MEAN and float(np.mean(vals>=THRESH))>=MIN_FRAC
            if ok:safe|=region
            accepted.append({'page':page,'region_id':cid,'pixels':int(region.sum()),'mean_probability':float(vals.mean()),'minimum_probability':float(vals.min()),'accepted':ok})
        require(np.all(~safe|m3),'SAFE_REMOVE outside M3'); m5=m3&~safe; corrected=telea(observed,m5.astype(np.uint8)*255); corrected[safe]=observed[safe]; require(np.array_equal(corrected[~m3],observed[~m3]),'outside authority drift')
        d.mkdir(parents=True,exist_ok=False)
        for name,a in [('observed.png',observed),('authoritative_seed.png',(seed>0).astype(np.uint8)*255),('m3.png',m3.astype(np.uint8)*255),('safe_remove.png',safe.astype(np.uint8)*255),('m5_remove.png',m5.astype(np.uint8)*255),('v7_8_baseline.png',baseline),('corrected.png',corrected)]:cv2.imwrite(str(d/name),a)
        np.save(d/'p_remove.npy',pmap); pd.DataFrame(accepted).to_csv(d/'safe_remove_regions.csv',index=False); frame['page']=page; frame['raw_remove']=raw; frame['p_remove']=p; frame['safe_remove']=safe[frame.y,frame.x]; part=TABLES/f'{page}_fresh_safe_remove.parquet'; frame.to_parquet(part,index=False); parts.append(part)
        row={'page':page,'status':'PASS','m3_pixels':int(m3.sum()),'safe_remove_pixels':int(safe.sum()),'m5_pixels':int(m5.sum()),'outside_authority_drift':0,'m5_formula_xor':int(np.logical_xor(m5,m3&~safe).sum()),'seconds':time.time()-t0,'corrected_sha256':array_hash(corrected)}; atomic_json(d/'page_manifest.json',row); page_rows.append(row); print(f'[{pos+1}/{len(roster)}] PASS {page}: SAFE_REMOVE={row["safe_remove_pixels"]:,}')
    except Exception as e:
        failure={'page':page,'status':'FAIL_CLOSED_M3','error':str(e),'traceback':traceback.format_exc(),'review_audit_path':r.review_audit_path}; atomic_json(FAILURES/f'{page}.json',failure); failures.append(failure); print('FAIL CLOSED',page,e)
        if STOP_AT_FIRST_PAGE_FAILURE:raise
page_df=pd.DataFrame(page_rows); fail_df=pd.DataFrame(failures); page_df.to_csv(TABLES/'page_results.csv',index=False); fail_df.to_csv(TABLES/'page_failures.csv',index=False)
if parts:pd.concat([pd.read_parquet(p) for p in parts],ignore_index=True).to_parquet(TABLES/'all_fresh_safe_remove_predictions.parquet',index=False)


## 6. Replay, verdict, visual queue and final upload package


In [ ]:
require(len(page_df)>0,'No page passed')
sample=page_df.iloc[0].page; rr=roster.set_index('page').loc[sample]; observed=read_gray(rr.observed_path); m3=read_gray(PAGES/sample/'m3.png')>0; safe=read_gray(PAGES/sample/'safe_remove.png')>0; m5=read_gray(PAGES/sample/'m5_remove.png')>0; saved=read_gray(PAGES/sample/'corrected.png'); replayed=telea(observed,m5.astype(np.uint8)*255); replayed[safe]=observed[safe]
replay_record={'page':sample,'m5_formula_exact':bool(np.array_equal(m5,m3&~safe)),'corrected_exact':bool(np.array_equal(saved,replayed))}; atomic_json(AUDIT/'deterministic_replay.json',replay_record)
checks2={'59r1_bundle_verified':True,'all_pages_accounted':len(page_df)+len(fail_df)==len(roster),'zero_page_failures':len(fail_df)==0,'zero_outside_drift':bool((page_df.outside_authority_drift==0).all()),'zero_m5_xor':bool((page_df.m5_formula_xor==0).all()),'replay_exact':all(replay_record.values()) if False else replay_record['m5_formula_exact'] and replay_record['corrected_exact'],'no_clean_truth':True,'safe_protect_disabled':True,'no_refit':True}
first=next((k for k,v in checks2.items() if not v),None); verdict='PASS_60R2_V9_0B_FRESH_SAFE_REMOVE' if first is None else 'FAIL_OR_PARTIAL_60R2_V9_0B'
record={'version':'60R2-v9.0B-fresh-safe-remove-HITL-59R1','verdict':verdict,'first_failed_gate':first,'checks':checks2,'resolved_pages':len(roster),'passed_pages':len(page_df),'failed_pages':len(fail_df),'policy_id':policy['policy_id'],'compatibility_id':policy['compatibility_id'],'safe_remove_total':int(page_df.safe_remove_pixels.sum()),'m3_total':int(page_df.m3_pixels.sum()),'m5_total':int(page_df.m5_pixels.sum()),'production_authority':False}; atomic_json(OUT/'notebook60r2_verdict.json',record)
queue=page_df.copy(); queue['safe_remove_fraction']=queue.safe_remove_pixels/np.maximum(queue.m3_pixels,1); queue.sort_values(['safe_remove_fraction','safe_remove_pixels'],ascending=False).to_csv(TABLES/'visual_review_queue.csv',index=False)
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'run':str(OUT),'version':record['version'],'bundle':str(BUNDLE),'policy_sha256':sha256_file(BUNDLE/'policy.json'),'schema_sha256':sha256_file(BUNDLE/'predictor_schema.json'),'model_sha256':sha256_file(BUNDLE/policy['model_file']),'calibrator_sha256':sha256_file(BUNDLE/policy['calibrator_file']),'roster_sha256':sha256_file(TABLES/'hitl_reviewed_roster.csv'),'environment':{'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}}; atomic_json(OUT/'experiment_identity.json',identity)
lines=['NOTEBOOK 60R2 / V9.0B FRESH SAFE_REMOVE HITL OPERATIONALISATION','',f'Verdict: {verdict}',f'Pages passed / failed: {len(page_df)} / {len(fail_df)}',f'59R1 policy: {policy["policy_id"]}',f'SAFE_REMOVE / M3 / M5: {record["safe_remove_total"]:,} / {record["m3_total"]:,} / {record["m5_total"]:,}','','59R1 joblib objects loaded from Drive and hash-verified.','No clean truth, refit, recalibration, SAFE_PROTECT or production authority.']; (OUT/'executive_summary.txt').write_text('
'.join(lines))
required=[OUT/'executive_summary.txt',OUT/'notebook60r2_verdict.json',OUT/'experiment_identity.json',AUDIT/'notebook59r1_source.json',AUDIT/'deterministic_replay.json',TABLES/'hitl_reviewed_roster.csv',TABLES/'page_results.csv',TABLES/'page_failures.csv',TABLES/'visual_review_queue.csv']
if (TABLES/'all_fresh_safe_remove_predictions.parquet').exists():required.append(TABLES/'all_fresh_safe_remove_predictions.parquet')
rows=[]
for src in required:
    require(src.exists(),f'Missing upload artifact {src}'); cat='tables' if src.parent==TABLES else 'audit' if src.parent==AUDIT else 'root'; dst=UPLOAD/cat/src.name; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst); rows.append({'relative_path':str(dst.relative_to(UPLOAD)),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False); (UPLOAD/'README.txt').write_text('60R2 uses the hash-verified Notebook 59R1 Drive bundle. Binary joblib files are intentionally not included. Research/HITL evidence only.
'); zip_path=UPLOAD/'60R2_V9_0B_review_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file() and p!=zip_path:z.write(p,p.relative_to(UPLOAD))
print('
'.join(lines)); print('Upload bundle:',zip_path)
